# Preprocessing Pipeline for snRNA-seq Data

**Module 00: Complete Preprocessing**

---

## Overview

Comprehensive preprocessing pipeline:

1. Quality Control - snRNA-seq appropriate thresholds
2. Normalization - 10,000 counts/cell
3. Feature Selection - 2,000 HVGs
4. Batch Correction - Harmony
5. Clustering - Leiden
6. Validation - Yang-style markers

**Data Layers:**
- `adata.X` - Log-normalized (active)
- `layers['counts']` - Raw counts (for DESeq2)
- `layers['scaled']` - Scaled data
- `adata.raw` - Full matrix

---

**Author:** Gerald Gaitos  
**Date:** December 2025  
**Pipeline:** Cellular Senescence Analysis

---

## Setup & Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════

import scanpy as sc
import scanpy.external as sce
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Check versions
print(f"scanpy: {sc.__version__}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION - Change DATASET line only
# ═══════════════════════════════════════════════════════════════════════════════

# ───────────────────────────────────────────────────────────────────────────────
# DATASET SELECTION
# ───────────────────────────────────────────────────────────────────────────────

DATASET = 'psychad_aging'  # ← CHANGE THIS LINE ONLY

# Options: 'psychad_aging', 'psychad_ad', 'psychencode', 'mathys', 'australian'

# ───────────────────────────────────────────────────────────────────────────────
# DATASET-SPECIFIC SETTINGS
# ───────────────────────────────────────────────────────────────────────────────

DATASET_CONFIG = {
    'psychad_aging': {
        'input_file': '/fs/scratch/PAS2598/data/psychad/aging_raw.h5ad',
        'batch_key': 'Sample',
        'cell_type_column': 'broad_class',
    },
    'psychad_ad': {
        'input_file': '/fs/scratch/PAS2598/data/psychad/ad_raw.h5ad',
        'batch_key': 'Sample',
        'cell_type_column': 'broad_class',
    },
    'psychencode': {
        'input_file': '/fs/scratch/PAS2598/data/psychencode/raw.h5ad',
        'batch_key': 'Donor',
        'cell_type_column': 'cell_type',
    },
    'mathys': {
        'input_file': '/fs/scratch/PAS2598/data/mathys/raw.h5ad',
        'batch_key': 'individual',
        'cell_type_column': 'broad.cell.type',
    },
    'australian': {
        'input_file': '/fs/scratch/PAS2598/data/australian/raw.h5ad',
        'batch_key': 'projid',
        'cell_type_column': 'celltype',
    },
}

# Get config for selected dataset
config = DATASET_CONFIG[DATASET]
INPUT_FILE = Path(config['input_file'])
BATCH_KEY = config['batch_key']
CELL_TYPE_COLUMN = config['cell_type_column']

# ───────────────────────────────────────────────────────────────────────────────
# QC PARAMETERS (snRNA-seq)
# ───────────────────────────────────────────────────────────────────────────────

MIN_GENES = 200
MAX_GENES = 8000
MAX_MT_PERCENT = 5  # Strict for nuclear RNA
MIN_CELLS = 3

# ───────────────────────────────────────────────────────────────────────────────
# PROCESSING PARAMETERS
# ───────────────────────────────────────────────────────────────────────────────

TARGET_SUM = 10000
N_TOP_GENES = 2000
N_PCS = 50
N_NEIGHBORS = 30
LEIDEN_RESOLUTION = 0.8
LEIDEN_FLAVOR = 'igraph'
LEIDEN_N_ITERATIONS = 2
PERFORM_SCALING = True
SCALE_MAX_VALUE = 10

# ───────────────────────────────────────────────────────────────────────────────
# PATHS
# ───────────────────────────────────────────────────────────────────────────────

BASE_DIR = Path('/fs/scratch/PAS2598/senescence_analysis')
OUTPUT_DIR = BASE_DIR / 'data' / 'processed'
FIGURES_DIR = BASE_DIR / 'figures' / '00_preprocessing' / DATASET

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / f'{DATASET}_preprocessed.h5ad'

# ───────────────────────────────────────────────────────────────────────────────
# RANDOM SEED
# ───────────────────────────────────────────────────────────────────────────────

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ───────────────────────────────────────────────────────────────────────────────
# CANONICAL MARKERS (Yang-style: 2-3 per type)
# ───────────────────────────────────────────────────────────────────────────────

CANONICAL_MARKERS = {
    'Excitatory': ['SLC17A7', 'SATB2'],
    'Inhibitory': ['GAD1', 'GAD2'],
    'Astrocyte': ['SLC1A2', 'GFAP', 'AQP4'],
    'Oligodendrocyte': ['MBP', 'MOBP', 'MOG'],
    'OPC': ['PDGFRA', 'CSPG4'],
    'Microglia': ['CX3CR1', 'TMEM119', 'CSF1R'],
    'Endothelial': ['CLDN5', 'FLT1'],
    'Pericyte': ['PDGFRB', 'RGS5'],
    'VLMC': ['DCN', 'COL1A1'],
    'VSMC': ['MYH11', 'TAGLN'],
}

# ───────────────────────────────────────────────────────────────────────────────
# DISPLAY CONFIGURATION
# ───────────────────────────────────────────────────────────────────────────────

print("="*80)
print("CONFIGURATION")
print("="*80)
print(f"\nDataset: {DATASET}")
print(f"Input: {INPUT_FILE}")
print(f"Batch key: {BATCH_KEY}")
print(f"Cell type column: {CELL_TYPE_COLUMN}")
print(f"\nQC: {MIN_GENES}-{MAX_GENES} genes/cell, <{MAX_MT_PERCENT}% MT")
print(f"Processing: {TARGET_SUM} counts/cell, {N_TOP_GENES} HVGs, {N_PCS} PCs")
print(f"Clustering: k={N_NEIGHBORS}, res={LEIDEN_RESOLUTION}")
print(f"\nOutput: {OUTPUT_FILE}")
print(f"Figures: {FIGURES_DIR}")
print("="*80)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE SETTINGS
# ═══════════════════════════════════════════════════════════════════════════════

sc.settings.set_figure_params(dpi=300, dpi_save=300, frameon=False, figsize=(6, 6))
sc.settings.figdir = FIGURES_DIR

# Publication-quality style
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 10,
    'axes.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

print(f"✓ Figure settings configured (300 DPI)")

---

## Step 1: Load Data

In [ ]:
print("="*80)
print("LOADING DATA")
print("="*80)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

print(f"\nLoading: {INPUT_FILE.name}...")
adata = sc.read_h5ad(INPUT_FILE)

print(f"\n✓ Data loaded")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")

# Verify batch key
if BATCH_KEY not in adata.obs.columns:
    raise ValueError(f"Batch key '{BATCH_KEY}' not found")
print(f"\n✓ Batch key '{BATCH_KEY}': {adata.obs[BATCH_KEY].nunique()} batches")

if CELL_TYPE_COLUMN and CELL_TYPE_COLUMN in adata.obs.columns:
    print(f"✓ Cell types '{CELL_TYPE_COLUMN}': {adata.obs[CELL_TYPE_COLUMN].nunique()} types")

---

## Step 2: Quality Control

In [ ]:
print("="*80)
print("QC METRICS")
print("="*80)

# Identify MT genes
adata.var['mt'] = adata.var_names.str.startswith('MT-')
print(f"\nMT genes: {adata.var['mt'].sum()}")

# Calculate QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

print(f"\nBefore filtering:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  Median genes/cell: {adata.obs['n_genes_by_counts'].median():.0f}")
print(f"  Median MT%: {adata.obs['pct_counts_mt'].median():.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'{DATASET} - QC Metrics Before Filtering', fontsize=14, fontweight='bold')

# Genes per cell
axes[0].hist(adata.obs['n_genes_by_counts'], bins=100, color='#4A90E2', alpha=0.7)
axes[0].axvline(MIN_GENES, color='red', linestyle='--', linewidth=2, label=f'Min: {MIN_GENES}')
axes[0].axvline(MAX_GENES, color='red', linestyle='--', linewidth=2, label=f'Max: {MAX_GENES}')
axes[0].set_xlabel('Genes per cell')
axes[0].set_ylabel('Cells')
axes[0].legend()

# Total counts
axes[1].hist(np.log10(adata.obs['total_counts']), bins=100, color='#50C878', alpha=0.7)
axes[1].set_xlabel('log₁₀(Counts per cell)')
axes[1].set_ylabel('Cells')

# MT %
axes[2].hist(adata.obs['pct_counts_mt'], bins=100, color='#E24A4A', alpha=0.7)
axes[2].axvline(MAX_MT_PERCENT, color='red', linestyle='--', linewidth=2, label=f'Max: {MAX_MT_PERCENT}%')
axes[2].set_xlabel('% Mitochondrial')
axes[2].set_ylabel('Cells')
axes[2].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_qc_before.pdf')
plt.show()
print(f"✓ QC plots saved")

In [ ]:
print("\nFiltering...")
n_cells_before = adata.n_obs
n_genes_before = adata.n_vars

# Filter genes
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)

# Filter cells
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
adata = adata[adata.obs['n_genes_by_counts'] <= MAX_GENES, :].copy()
adata = adata[adata.obs['pct_counts_mt'] < MAX_MT_PERCENT, :].copy()

n_cells_removed = n_cells_before - adata.n_obs
n_genes_removed = n_genes_before - adata.n_vars

print(f"\n✓ Filtering complete")
print(f"  Cells: {n_cells_before:,} → {adata.n_obs:,} (removed {n_cells_removed:,}, {n_cells_removed/n_cells_before*100:.1f}%)")
print(f"  Genes: {n_genes_before:,} → {adata.n_vars:,} (removed {n_genes_removed:,}, {n_genes_removed/n_genes_before*100:.1f}%)")

---

## Step 3: Data Layers & Normalization

In [ ]:
print("="*80)
print("DATA LAYERS")
print("="*80)

# Save raw counts
print("\n1. Saving raw counts...")
adata.layers['counts'] = adata.X.copy()
print("   ✓ layers['counts'] - for DESeq2")

# Normalize
print(f"\n2. Normalizing to {TARGET_SUM} counts/cell...")
sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
adata.layers['normalized'] = adata.X.copy()
print("   ✓ layers['normalized']")

# Log transform
print("\n3. Log transforming...")
sc.pp.log1p(adata)
adata.layers['log1p'] = adata.X.copy()
print("   ✓ layers['log1p'] (active layer)")

print("\n✓ Data layers created")

---

## Step 4: Feature Selection

In [ ]:
print("="*80)
print("HIGHLY VARIABLE GENES")
print("="*80)

print(f"\nSelecting top {N_TOP_GENES} HVGs...")
sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES, flavor='seurat', subset=False)

n_hvgs = adata.var['highly_variable'].sum()
print(f"✓ {n_hvgs:,} HVGs identified (all genes kept)")

# Plot
sc.pl.highly_variable_genes(adata, save=f'_{DATASET}_hvgs.pdf')

---

## Step 5: Scaling & PCA

In [ ]:
print("="*80)
print("SCALING & PCA")
print("="*80)

# Save raw
print("\n1. Saving full matrix to adata.raw...")
adata.raw = adata
print("   ✓ adata.raw saved")

# Scale
if PERFORM_SCALING:
    print("\n2. Scaling data...")
    sc.pp.scale(adata, max_value=SCALE_MAX_VALUE)
    adata.layers['scaled'] = adata.X.copy()
    # Restore log-normalized
    adata.X = adata.layers['log1p'].copy()
    print("   ✓ layers['scaled'] - log-normalized restored to active")

# PCA
print(f"\n3. Computing PCA ({N_PCS} components)...")
sc.tl.pca(adata, n_comps=N_PCS, use_highly_variable=True, random_state=RANDOM_SEED)
variance = adata.uns['pca']['variance_ratio'].sum()
print(f"   ✓ PCA complete (variance explained: {variance:.1%})")

# Plot
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True, save=f'_{DATASET}_pca.pdf')

---

## Step 6: Batch Correction

In [ ]:
print("="*80)
print("HARMONY BATCH CORRECTION")
print("="*80)

print(f"\nBatch key: {BATCH_KEY}")
print(f"Batches: {adata.obs[BATCH_KEY].nunique()}")

sce.pp.harmony_integrate(
    adata,
    key=BATCH_KEY,
    basis='X_pca',
    adjusted_basis='X_pca_harmony',
    random_state=RANDOM_SEED
)

print("\n✓ Harmony complete")

---

## Step 7: UMAP & Clustering

In [ ]:
print("="*80)
print("NEIGHBORS, UMAP & CLUSTERING")
print("="*80)

# Neighbors
print(f"\n1. Computing neighbors (k={N_NEIGHBORS})...")
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS, use_rep='X_pca_harmony', random_state=RANDOM_SEED)
print("   ✓ Neighbor graph computed")

# UMAP
print("\n2. Computing UMAP...")
sc.tl.umap(adata, random_state=RANDOM_SEED)
print("   ✓ UMAP complete")

# Leiden
print(f"\n3. Leiden clustering (res={LEIDEN_RESOLUTION})...")
sc.tl.leiden(
    adata,
    resolution=LEIDEN_RESOLUTION,
    flavor=LEIDEN_FLAVOR,
    n_iterations=LEIDEN_N_ITERATIONS,
    random_state=RANDOM_SEED
)
n_clusters = adata.obs['leiden'].nunique()
print(f"   ✓ {n_clusters} clusters identified")

# Plot
sc.pl.umap(adata, color=BATCH_KEY, title='Batch Correction', save=f'_{DATASET}_batch.pdf')
sc.pl.umap(adata, color='leiden', legend_loc='on data', save=f'_{DATASET}_clusters.pdf')

---

## Step 8: Cell Type Validation

In [ ]:
print("="*80)
print("CELL TYPE VALIDATION")
print("="*80)

if CELL_TYPE_COLUMN and CELL_TYPE_COLUMN in adata.obs.columns:
    # Filter markers
    markers_filtered = {
        ct: [g for g in genes if g in adata.var_names]
        for ct, genes in CANONICAL_MARKERS.items()
    }
    markers_filtered = {k: v for k, v in markers_filtered.items() if len(v) > 0}
    
    print(f"\nMarkers available: {sum(len(v) for v in markers_filtered.values())}")
    for ct, genes in markers_filtered.items():
        print(f"  {ct}: {', '.join(genes)}")
    
    # Cell type order
    celltype_order = [
        'Excitatory', 'Inhibitory',
        'Astrocyte', 'Oligodendrocyte', 'OPC',
        'Microglia',
        'Endothelial', 'Pericyte', 'VLMC', 'VSMC',
    ]
    available = adata.obs[CELL_TYPE_COLUMN].unique()
    celltype_order = [ct for ct in celltype_order if ct in available]
    celltype_order.extend(sorted([ct for ct in available if ct not in celltype_order]))
    
    # Dotplot
    print("\nCreating Yang-style validation dotplot...")
    fig, ax = plt.subplots(figsize=(8, 5))
    sc.pl.dotplot(
        adata,
        var_names=markers_filtered,
        groupby=CELL_TYPE_COLUMN,
        categories_order=celltype_order,
        standard_scale='var',
        cmap='Reds',
        dot_min=0.1,
        dot_max=0.9,
        ax=ax,
        show=False
    )
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{DATASET}_markers.pdf')
    plt.savefig(FIGURES_DIR / f'{DATASET}_markers.png')
    plt.show()
    
    # Cell type UMAP
    sc.pl.umap(adata, color=CELL_TYPE_COLUMN, legend_loc='right margin', save=f'_{DATASET}_celltypes.pdf')
    
    print("\n✓ Validation complete")
else:
    print("\n⚠ Cell type column not found - validation skipped")

---

## Step 9: Save

In [ ]:
print("="*80)
print("SAVING")
print("="*80)

print(f"\nSaving to: {OUTPUT_FILE}")
adata.write(OUTPUT_FILE)

file_size_gb = OUTPUT_FILE.stat().st_size / 1e9
print(f"\n✓ Saved ({file_size_gb:.2f} GB)")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  Clusters: {adata.obs['leiden'].nunique()}")

---

## Summary

In [ ]:
print("="*80)
print("✓ PREPROCESSING COMPLETE")
print("="*80)

print(f"\nDataset: {DATASET}")
print(f"\nFinal dimensions:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  HVGs: {adata.var['highly_variable'].sum():,}")
print(f"  Clusters: {adata.obs['leiden'].nunique()}")

print(f"\nData layers:")
print(f"  adata.X - log-normalized (active)")
print(f"  layers['counts'] - raw counts (for DESeq2)")
print(f"  layers['normalized'] - normalized")
print(f"  layers['log1p'] - log-normalized")
if PERFORM_SCALING:
    print(f"  layers['scaled'] - scaled")
print(f"  adata.raw - full matrix")

print(f"\nOutputs:")
print(f"  Data: {OUTPUT_FILE.name}")
print(f"  Figures: {FIGURES_DIR.name}/")

print(f"\nReady for:")
print(f"  → Module 01: Senescence scoring")
print(f"  → Module 02: Glial subclustering")
print(f"  → Module 06: DEG analysis")

print("\n" + "="*80)